# 7.4 · Randomized algorithms

*Algorithm design* · 17 exercises

Use randomness to shuffle, sample, balance and estimate, and to test primes fast with a tiny chance of error.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ch-7.4) · [Answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ch-7.4) · [Concept guide](https://bloomcode.github.io/dsa/guide/#ch-7.4)

**How to work:** run the setup cell once. For each exercise, replace the stub in its code cell, run the cell, then run the test cell under it. Later exercises often reuse earlier ones, so go in order. Stuck? Each exercise links to its tested solution in the answer key.

In [ ]:
# Setup: run once.
import bisect
import itertools
import math
import random
from collections import Counter

try:
    import pytest
except ImportError:  # the tests only need pytest.raises; a small stand-in keeps the notebook self-contained
    class _Raises:
        def __init__(self, kind):
            self.kind = kind

        def __enter__(self):
            return self

        def __exit__(self, kind, value, tb):
            if kind is None:
                raise AssertionError(f"did not raise {self.kind.__name__}")
            return issubclass(kind, self.kind)

    class pytest:  # noqa: N801
        raises = _Raises

### 7.4.1 · Randomized quicksort  `Core`

**Task.** Write `random_quicksort(arr, rng)` returning a new sorted list, choosing each pivot uniformly at random. Count comparisons on sorted input and compare with 22.6.

**Example.** `random_quicksort(list(range(10)), random.Random(1))` -> `[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]`

**Big O.** O(N log N) expected on every input, including sorted ones. The worst case still exists but needs extraordinarily bad luck.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-7-4-1) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-7-4-1)

In [ ]:
# 7.4.1 Randomized quicksort
def random_quicksort(arr, rng):
    raise NotImplementedError

In [ ]:
def test_ex_7_4_1():
    rng = random.Random(0)
    for _ in range(60):
        data = [rng.randint(-30, 30) for _ in range(rng.randint(0, 40))]
        assert random_quicksort(data, random.Random(3)) == sorted(data)
    assert random_quicksort(list(range(10)), random.Random(1)) == list(range(10))


test_ex_7_4_1()
print('7.4.1 passed')

### 7.4.2 · Reproducible randomness with seeds  `Core`

**Task.** Write `sample_with_seed(seed, population, k)` that returns the same k-item sample every time for the same seed, using its own `random.Random(seed)` rather than the global generator.

**Example.** `sample_with_seed(42, range(100), 3) == sample_with_seed(42, range(100), 3)` -> `True`

**Big O.** O(k).

**Note.** A private `Random` instance keeps your results reproducible even when other code also draws random numbers.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-7-4-2) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-7-4-2)

In [ ]:
# 7.4.2 Reproducible randomness with seeds
def sample_with_seed(seed, population, k):
    raise NotImplementedError

In [ ]:
def test_ex_7_4_2():
    assert sample_with_seed(42, range(100), 3) == sample_with_seed(42, range(100), 3)
    assert len(sample_with_seed(42, range(100), 3)) == 3


test_ex_7_4_2()
print('7.4.2 passed')

### 7.4.3 · A linear congruential generator  `Core`

**Task.** Write a generator `lcg(seed, a=1103515245, c=12345, m=2**31)` yielding `x = (a * x + c) % m` forever. Then find the period of the tiny generator `a=5, c=3, m=16`.

**Example.** first three values from `lcg(1, 5, 3, 16)` -> `[8, 11, 10]`; its period is `16`

**Big O.** O(1) per number. This is a PRNG: fully determined by the seed, which is why it is reproducible and why it is not secure.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-7-4-3) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-7-4-3)

In [ ]:
# 7.4.3 A linear congruential generator
def lcg(seed, a=1103515245, c=12345, m=2**31):
    raise NotImplementedError


def lcg_period(seed, a, c, m):
    raise NotImplementedError

In [ ]:
def test_ex_7_4_3():
    g = lcg(1, 5, 3, 16)
    assert [next(g) for _ in range(3)] == [8, 11, 10]
    assert lcg_period(1, 5, 3, 16) == 16


test_ex_7_4_3()
print('7.4.3 passed')

### 7.4.4 · Fisher-Yates shuffle  `Core`

**Task.** Write `fisher_yates(arr, rng)` that shuffles in place: for i from the last index down to 1, swap arr[i] with arr[j] for a random j in 0..i.

**Example.** `fisher_yates([1, 2, 3, 4, 5], random.Random(7))` returns some ordering of 1..5, the same one every time for seed 7

**Big O.** O(N) time, O(1) extra space, and every permutation is equally likely.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-7-4-4) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-7-4-4)

In [ ]:
# 7.4.4 Fisher-Yates shuffle
def fisher_yates(arr, rng):
    raise NotImplementedError

In [ ]:
def test_ex_7_4_4():
    a = fisher_yates([1, 2, 3, 4, 5], random.Random(7))
    assert sorted(a) == [1, 2, 3, 4, 5]
    assert fisher_yates([1, 2, 3, 4, 5], random.Random(7)) == a


test_ex_7_4_4()
print('7.4.4 passed')

### 7.4.5 · Prove the naive shuffle is biased  `Practice`

**Task.** The naive shuffle swaps each position with any position 0..n-1. Write `naive_distribution(n)` and `fisher_yates_distribution(n)` that enumerate every possible sequence of random choices and count the resulting permutations.

**Example.** For n = 3: Fisher-Yates gives each of the 6 permutations exactly once; naive spreads 27 outcomes as `{4, 5, 5, 5, 4, 4}`

**Big O.** O(n^n) and O(n!) sequences: exact enumeration, no randomness needed.

**Note.** 27 equally likely outcomes cannot split evenly over 6 permutations, so the naive shuffle must be biased.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-7-4-5) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-7-4-5)

In [ ]:
# 7.4.5 Prove the naive shuffle is biased
def naive_distribution(n):
    raise NotImplementedError


def fisher_yates_distribution(n):
    raise NotImplementedError

In [ ]:
def test_ex_7_4_5():
    fy = fisher_yates_distribution(3)
    assert len(fy) == 6 and set(fy.values()) == {1}
    nv = naive_distribution(3)
    assert sum(nv.values()) == 27 and sorted(nv.values()) == [4, 4, 4, 5, 5, 5]
    assert set(fisher_yates_distribution(4).values()) == {1}


test_ex_7_4_5()
print('7.4.5 passed')

### 7.4.6 · Randomize before building a BST  `Core`

**Task.** Write `bst_height(keys)` that inserts keys in order into a plain BST and returns its height. Compare sorted keys with the same keys shuffled.

**Example.** `bst_height(list(range(200)))` -> `200`; shuffled with `random.Random(3)` -> about 15 to 20

**Big O.** Height O(N) for sorted input, O(log N) expected after shuffling.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-7-4-6) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-7-4-6)

In [ ]:
# 7.4.6 Randomize before building a BST
def bst_height(keys):
    raise NotImplementedError

In [ ]:
def test_ex_7_4_6():
    assert bst_height(list(range(200))) == 200
    keys = list(range(200))
    random.Random(3).shuffle(keys)
    assert bst_height(keys) < 30


test_ex_7_4_6()
print('7.4.6 passed')

### 7.4.7 · Reservoir sampling  `Practice`

**Task.** Write `reservoir(stream, k, rng)` that picks k items uniformly from a stream of unknown length in one pass: keep the first k, then replace a random slot with item i (0-based) with probability k / (i + 1).

**Example.** `reservoir(range(1_000_000), 3, random.Random(0))` -> 3 values, each equally likely to be any of the million

**Big O.** O(N) time, O(k) memory.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-7-4-7) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-7-4-7)

In [ ]:
# 7.4.7 Reservoir sampling
def reservoir(stream, k, rng):
    raise NotImplementedError

In [ ]:
def test_ex_7_4_7():
    s = reservoir(range(10**5), 3, random.Random(0))
    assert len(s) == 3 and len(set(s)) == 3
    firsts = [reservoir(range(10), 1, random.Random(i))[0] for i in range(5000)]
    assert all(350 < firsts.count(v) < 650 for v in range(10))


test_ex_7_4_7()
print('7.4.7 passed')

### 7.4.8 · Load balancing with random choices  `Core`

**Task.** Write `max_load_random(jobs, servers, rng)` assigning each job to a random server, and `max_load_two_choices(jobs, servers, rng)` picking two random servers and using the less loaded one. Return the busiest server's load.

**Example.** With 10,000 jobs on 1,000 servers and seed 5, two choices gives a clearly lower maximum than one choice

**Big O.** O(J) for J jobs. The "power of two choices" drops the expected maximum excess from about log N / log log N to log log N.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-7-4-8) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-7-4-8)

In [ ]:
# 7.4.8 Load balancing with random choices
def max_load_random(jobs, servers, rng):
    raise NotImplementedError


def max_load_two_choices(jobs, servers, rng):
    raise NotImplementedError

In [ ]:
def test_ex_7_4_8():
    one = max_load_random(10000, 1000, random.Random(5))
    two = max_load_two_choices(10000, 1000, random.Random(5))
    assert two < one and two <= 14


test_ex_7_4_8()
print('7.4.8 passed')

### 7.4.9 · Weighted random choice  `Practice`

**Task.** Write `make_weighted_picker(weights, rng)` that precomputes prefix sums and returns a function picking index i with probability weights[i] / total, using `bisect`.

**Example.** weights `[1, 3, 6]` -> over 10,000 picks index 2 comes up about 60% of the time

**Big O.** O(N) setup, O(log N) per pick.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-7-4-9) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-7-4-9)

In [ ]:
# 7.4.9 Weighted random choice
def make_weighted_picker(weights, rng):
    raise NotImplementedError

In [ ]:
def test_ex_7_4_9():
    pick = make_weighted_picker([1, 3, 6], random.Random(2))
    picks = [pick() for _ in range(10000)]
    assert 5600 < picks.count(2) < 6400 and set(picks) == {0, 1, 2}


test_ex_7_4_9()
print('7.4.9 passed')

### 7.4.10 · Estimate pi by throwing darts  `Core`

**Task.** Write `estimate_pi(samples, rng)`: pick random points in the unit square and return 4 times the fraction that land inside the quarter circle.

**Example.** `estimate_pi(100_000, random.Random(0))` -> about `3.14` (within 0.02)

**Big O.** O(S) samples; the error shrinks like 1 / sqrt(S), so 100 times more samples buy one more correct digit.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-7-4-10) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-7-4-10)

In [ ]:
# 7.4.10 Estimate pi by throwing darts
def estimate_pi(samples, rng):
    raise NotImplementedError

In [ ]:
def test_ex_7_4_10():
    assert abs(estimate_pi(100_000, random.Random(0)) - 3.14159) < 0.02


test_ex_7_4_10()
print('7.4.10 passed')

### 7.4.11 · Averages from a random sample  `Core`

**Task.** Write `sampled_mean(data, k, rng)` estimating the mean of a huge list from k random positions (with replacement), and `exact_mean(data)` for comparison.

**Example.** one million values, k = 2,000 -> the estimate lands within about 2% of the true mean

**Big O.** O(k) instead of O(N); accuracy depends on k and the data's spread, not on N.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-7-4-11) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-7-4-11)

In [ ]:
# 7.4.11 Averages from a random sample
def sampled_mean(data, k, rng):
    raise NotImplementedError


def exact_mean(data):
    raise NotImplementedError

In [ ]:
def test_ex_7_4_11():
    rng = random.Random(4)
    data = [rng.randint(0, 1000) for _ in range(10**6)]
    assert abs(sampled_mean(data, 2000, random.Random(1)) / exact_mean(data) - 1) < 0.04
    assert exact_mean([1, 2, 3, 6]) == 3


test_ex_7_4_11()
print('7.4.11 passed')

### 7.4.12 · Monte Carlo versus Las Vegas  `Core`

**Task.** In a list where half the entries are 1, write `las_vegas_find_one(arr, rng)` (probe random positions until a 1 appears; always right, time varies) and `monte_carlo_find_one(arr, rng, tries)` (give up after `tries` probes and return -1; time fixed, may fail).

**Example.** both return an index holding 1; Monte Carlo with 20 tries fails with probability (1/2)^20, about one in a million

**Big O.** Las Vegas: 2 probes expected. Monte Carlo: exactly `tries` probes at most.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-7-4-12) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-7-4-12)

In [ ]:
# 7.4.12 Monte Carlo versus Las Vegas
def las_vegas_find_one(arr, rng):
    raise NotImplementedError


def monte_carlo_find_one(arr, rng, tries):
    raise NotImplementedError

In [ ]:
def test_ex_7_4_12():
    arr = [i % 2 for i in range(1000)]
    assert arr[las_vegas_find_one(arr, random.Random(2))] == 1
    i = monte_carlo_find_one(arr, random.Random(2), 20)
    assert i == -1 or arr[i] == 1
    assert monte_carlo_find_one([0] * 10, random.Random(2), 20) == -1


test_ex_7_4_12()
print('7.4.12 passed')

### 7.4.13 · The trial-division baseline  `Core`

**Task.** Write `is_prime_trial(n)` checking divisors up to sqrt(n). Time it on a 12-digit prime and note why it is too slow for 100-digit numbers.

**Example.** `is_prime_trial(999_999_000_001)` -> `True` after about 500,000 divisions

**Big O.** O(sqrt N): exponential in the number of digits.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-7-4-13) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-7-4-13)

In [ ]:
# 7.4.13 The trial-division baseline
def is_prime_trial(n):
    raise NotImplementedError

In [ ]:
def test_ex_7_4_13():
    assert is_prime_trial(999_999_000_001) and not is_prime_trial(1) and not is_prime_trial(999_999_000_001 * 3)
    assert [n for n in range(20) if is_prime_trial(n)] == [2, 3, 5, 7, 11, 13, 17, 19]


test_ex_7_4_13()
print('7.4.13 passed')

### 7.4.14 · Fermat's little theorem, one base  `Core`

**Task.** If n is prime, `pow(a, n - 1, n) == 1` for every a not divisible by n. Write `fermat_witness(a, n)` returning True when a proves n composite.

**Example.** `fermat_witness(2, 15)` -> `True` (15 is composite); `fermat_witness(2, 13)` -> `False`

**Big O.** O(log n) multiplications with three-argument pow.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-7-4-14) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-7-4-14)

In [ ]:
# 7.4.14 Fermat's little theorem, one base
def fermat_witness(a, n):
    raise NotImplementedError

In [ ]:
def test_ex_7_4_14():
    assert fermat_witness(2, 15) and not fermat_witness(2, 13) and not fermat_witness(2, 561)


test_ex_7_4_14()
print('7.4.14 passed')

### 7.4.15 · Fermat's primality test  `Core`

**Task.** Write `fermat_test(n, k, rng)` trying k random bases in [2, n - 2] and answering "probably prime" (True) if none is a witness. Then show 561, a Carmichael number, fools every base coprime to it.

**Example.** `fermat_test(10**9 + 7, 20, rng)` -> `True`; `fermat_test(1_000_000_007 * 998_244_353, 20, rng)` -> `False`; `fermat_witness(2, 561)` -> `False` although 561 = 3 x 11 x 17

**Big O.** O(k log n). A Monte Carlo algorithm: "composite" is always right, "prime" can be wrong.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-7-4-15) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-7-4-15)

In [ ]:
# 7.4.15 Fermat's primality test
def fermat_test(n, k, rng):
    raise NotImplementedError

In [ ]:
def test_ex_7_4_15():
    rng = random.Random(6)
    assert fermat_test(10**9 + 7, 20, rng) and not fermat_test(1_000_000_007 * 998_244_353, 20, rng)


test_ex_7_4_15()
print('7.4.15 passed')

### 7.4.16 · Miller-Rabin, which Carmichael numbers cannot fool  `Stretch`

**Task.** Write `miller_rabin(n, bases=(2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37))`: write n - 1 = d x 2^s, and for each base check whether `a^d` is 1 or reaches n - 1 by repeated squaring. With these 12 bases it is exact for every n below 3.3 x 10^24.

**Example.** `miller_rabin(561)` -> `False`; `miller_rabin(2**61 - 1)` -> `True`

**Big O.** O(k log^3 n) bit operations; effectively instant for 64-bit numbers.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-7-4-16) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-7-4-16)

In [ ]:
# 7.4.16 Miller-Rabin, which Carmichael numbers cannot fool
def miller_rabin(n, bases=(2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37)):
    raise NotImplementedError

In [ ]:
def test_ex_7_4_16():
    assert miller_rabin(999_999_000_001)
    assert not miller_rabin(561) and miller_rabin(2**61 - 1) and not miller_rabin(1)
    for n in range(2, 3000):
        assert miller_rabin(n) == all(n % d for d in range(2, int(n ** 0.5) + 1))
    for c in (561, 1105, 1729, 2465, 2821, 6601):
        assert not miller_rabin(c)


test_ex_7_4_16()
print('7.4.16 passed')

### 7.4.17 · Simulate, then check against the formula  `Practice`

**Task.** Write `birthday_simulation(people, trials, rng)` estimating the chance that two of `people` share a birthday (365 days), and `birthday_exact(people)` from the product formula.

**Example.** 23 people -> exact `0.507...`; 20,000 simulated trials land within 0.02 of it

**Big O.** O(trials x people). When no formula exists, simulation is how you get the number.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-7-4-17) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-7-4-17)

In [ ]:
# 7.4.17 Simulate, then check against the formula
def birthday_simulation(people, trials, rng):
    raise NotImplementedError


def birthday_exact(people):
    raise NotImplementedError

In [ ]:
def test_ex_7_4_17():
    assert abs(birthday_exact(23) - 0.5073) < 1e-3
    assert abs(birthday_simulation(23, 20000, random.Random(3)) - 0.5073) < 0.02


test_ex_7_4_17()
print('7.4.17 passed')